<h2>Data Cleaning and Pre- Processing</h2>

<h4>Data was already cleaned in the previous process</h4>

Loading all the libraries

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

Loading the cleaned data

In [100]:
data1= pd.read_csv('./Integrated Dataset_bhavyaVerma.csv')

Gettin info and shape of the data

In [101]:
print(data1.shape)
print(data1.info())

(48120, 12)
<class 'pandas.DataFrame'>
RangeIndex: 48120 entries, 0 to 48119
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   DateTime        48120 non-null  str    
 1   Date            48120 non-null  str    
 2   TimeStamp       48120 non-null  str    
 3   Junction        48120 non-null  int64  
 4   Vehicles        48120 non-null  int64  
 5   ID              48120 non-null  int64  
 6   Category        48120 non-null  str    
 7   Event Name      48120 non-null  str    
 8   Venue/Location  48120 non-null  str    
 9   Rain            48120 non-null  float64
 10  Temp Max        48120 non-null  float64
 11  Temp Min        48120 non-null  float64
dtypes: float64(3), int64(3), str(6)
memory usage: 4.4 MB
None


In [102]:
junctions= data1['Junction'].value_counts()
print(junctions)

IDs= data1['ID'].value_counts()
print(IDs)

categories= data1['Category'].value_counts()
print(categories)

events= data1['Event Name'].value_counts()
print(events)

loc= data1['Venue/Location'].value_counts()
print(loc)


Junction
1    14592
2    14592
3    14592
4     4344
Name: count, dtype: int64
ID
20151101001    1
20151101011    1
20151101021    1
20151101031    1
20151101041    1
              ..
20170630194    1
20170630204    1
20170630214    1
20170630224    1
20170630234    1
Name: count, Length: 48120, dtype: int64
Category
No Category          44448
Cultural & Social     1464
Music & Arts          1152
Tech & Business       1056
Name: count, dtype: int64
Event Name
No Event                           44448
Chitra Santhe                        168
Republic Day Flower Show             168
Kadalekai Parishe                    144
Pravasi Bharatiya Divas               96
Anoushka Shankar                      96
Dream Theater                         96
Aero India 2017                       96
Ugadi Feasts                          96
Ramanavami Music Festival             96
ReactFoo                              96
Namma Bengaluru Awards                72
IoT Next                              72
Und

Aggregating the data for each junction 

In [ ]:
data1['DateTime']= data1['Date'] + ' ' + data1['TimeStamp']
data1['time_obj']= pd.to_datetime(data1['DateTime'], format= "%Y-%m-%d %H:%M:%S")

time_grouped= data1.groupby(['Junction', pd.Grouper(key= 'time_obj', freq= 'h')]).agg({'Vehicles': 'sum',
                                                                                        'Rain': 'mean',
                                                                                        'Temp Max': 'mean',
                                                                                        'Temp Min': 'mean',
                                                                                        'Event Name': 'first'}).reset_index()
time_grouped['TimeStamp']= pd.to_datetime(time_grouped['time_obj']).dt.time
time_grouped

,Junction,time_obj,Vehicles,Rain,Temp Max,Temp Min,Event Name,TimeStamp
0,1,2015-11-01 00:00:00,15,0.0,28.602917,20.638700,No Event,00:00:00
1,1,2015-11-01 01:00:00,13,0.0,28.602917,20.638700,No Event,01:00:00
2,1,2015-11-01 02:00:00,10,0.0,28.602917,20.638700,No Event,02:00:00
3,1,2015-11-01 03:00:00,7,0.0,28.602917,20.638700,No Event,03:00:00
4,1,2015-11-01 04:00:00,9,0.0,28.602917,20.638700,No Event,04:00:00
...,...,...,...,...,...,...,...,...
48115,4,2017-06-30 19:00:00,11,0.0,30.073847,21.617704,No Event,19:00:00
48116,4,2017-06-30 20:00:00,30,0.0,30.073847,21.617704,No Event,20:00:00
48117,4,2017-06-30 21:00:00,16,0.0,30.073847,21.617704,No Event,21:00:00
48118,4,2017-06-30 22:00:00,22,0.0,30.073847,21.617704,No Event,22:00:00


Feature Engineering and Feature importance

In [ ]:
# Feature Engineering
# Time-based features
time_grouped['hour'] = time_grouped['time_obj'].dt.hour
time_grouped['day_of_week'] = time_grouped['time_obj'].dt.dayofweek
time_grouped['month'] = time_grouped['time_obj'].dt.month
time_grouped['is_weekend'] = time_grouped['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

time_grouped['is_special_event'] = time_grouped['Event Name'].apply(
    lambda x: 0 if str(x).lower() == 'no event' else 1
)

for i in [1, 2, 3]:
    time_grouped[f'vehicles_lag_{i}'] = time_grouped.groupby('Junction')['Vehicles'].shift(i)

time_grouped = time_grouped.dropna()

# Normalize/Standardize data
scaler = StandardScaler()
cols_to_scale = ['Vehicles', 'Rain', 'Temp Max', 'Temp Min']
time_grouped[[f'{col}_scaled' for col in cols_to_scale]] = scaler.fit_transform(time_grouped[cols_to_scale])

# Evaluate feature importance
feature_cols = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_special_event', 
                'vehicles_lag_1', 'vehicles_lag_2', 'vehicles_lag_3', 'Rain', 'Temp Max', 'Temp Min']
X = time_grouped[feature_cols]
y = time_grouped['Vehicles']

# Feature importance from Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

# Output final results
print(importances)
time_grouped.to_csv('Processed_Traffic_Data.csv', index=False)

RandomForestRegressor(random_state=42)
vehicles_lag_1      0.943694
hour                0.019183
vehicles_lag_2      0.008848
vehicles_lag_3      0.008175
Temp Min            0.005579
Temp Max            0.005515
day_of_week         0.002971
month               0.002833
Rain                0.002051
is_weekend          0.000771
is_special_event    0.000379
dtype: float64
